# 🤖 IMDB Sentiment Analysis - Model Development & Comparison

**Objective:** Build and compare 4 different models for sentiment classification

**Models to Compare:**
1. Logistic Regression (Traditional ML)
2. Support Vector Machine (Traditional ML)
3. Random Forest (Ensemble Method)
4. BERT Fine-tuning (Deep Learning)

**Evaluation Metrics:** Accuracy, Precision, Recall, F1-Score, Confusion Matrix

---

In [ ]:
# Core ML Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from tqdm.notebook import tqdm
import time
import pickle
import joblib

# Scikit-learn
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                           f1_score, confusion_matrix, classification_report,
                           roc_curve, auc, precision_recall_curve)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Imbalance handling
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline as ImbPipeline

# Deep Learning (BERT)
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                         Trainer, TrainingArguments, AdamW)
from datasets import Dataset as HFDataset

# Set style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
warnings.filterwarnings('ignore')

# Check GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🔥 Using device: {device}")

print("✅ All libraries imported successfully!")

## 📂 Load Processed Data

In [ ]:
# Load the processed data
try:
    df = pd.read_csv('processed_imdb_data.csv')
    print(f"✅ Loaded processed data: {df.shape}")
except FileNotFoundError:
    print("⚠️ Processed data not found. Loading and processing raw data...")
    # Load raw data and process it
    df_raw = pd.read_csv('archive/IMDB Dataset.csv')
    # Apply preprocessing (simplified version)
    from nltk.corpus import stopwords
    import re
    
    stop_words = set(stopwords.words('english'))
    
    def basic_preprocess(text):
        text = re.sub(r'<.*?>', '', text)
        text = re.sub(r'[^a-zA-Z\s]', '', text.lower())
        text = ' '.join([word for word in text.split() if word not in stop_words and len(word) > 2])
        return text
    
    df_raw['processed_review'] = df_raw['review'].progress_apply(basic_preprocess)
    df_raw['sentiment_encoded'] = df_raw['sentiment'].map({'negative': 0, 'positive': 1})
    df = df_raw[['processed_review', 'sentiment', 'sentiment_encoded']].copy()
    df.to_csv('processed_imdb_data.csv', index=False)

# Display sample
print("\n📋 Sample Data:")
df.head()

## 🎯 Train-Test Split & Feature Engineering

In [ ]:
# Prepare features and target
X = df['processed_review']
y = df['sentiment_encoded']

# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"📊 Data Split:")
print(f"Training set: {len(X_train):,} samples")
print(f"Test set: {len(X_test):,} samples")
print(f"Training sentiment distribution: {y_train.value_counts().to_dict()}")
print(f"Test sentiment distribution: {y_test.value_counts().to_dict()}")

# Initialize TF-IDF Vectorizer
tfidf_vectorizer = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    stop_words='english'
)

print(f"\n🔤 TF-IDF Vectorizer initialized with {tfidf_vectorizer.max_features:,} features")

## 🏗️ Model 1: Logistic Regression

In [ ]:
class ModelEvaluator:
    """Comprehensive model evaluation class."""
    
    def __init__(self, model_name):
        self.model_name = model_name
        self.training_time = 0
        self.prediction_time = 0
        
    def train_and_predict(self, pipeline, X_train, X_test, y_train):
        """Train model and make predictions."""
        start_time = time.time()
        pipeline.fit(X_train, y_train)
        self.training_time = time.time() - start_time
        
        start_time = time.time()
        y_pred = pipeline.predict(X_test)
        y_pred_proba = pipeline.predict_proba(X_test)[:, 1]
        self.prediction_time = time.time() - start_time
        
        return y_pred, y_pred_proba, pipeline
    
    def evaluate_model(self, y_true, y_pred, y_pred_proba):
        """Calculate all evaluation metrics."""
        metrics = {
            'accuracy': accuracy_score(y_true, y_pred),
            'precision': precision_score(y_true, y_pred),
            'recall': recall_score(y_true, y_pred),
            'f1_score': f1_score(y_true, y_pred)
        }
        
        # ROC AUC
        fpr, tpr, _ = roc_curve(y_true, y_pred_proba)
        metrics['roc_auc'] = auc(fpr, tpr)
        
        # PR AUC
        precision, recall, _ = precision_recall_curve(y_true, y_pred_proba)
        metrics['pr_auc'] = auc(recall, precision)
        
        return metrics, confusion_matrix(y_true, y_pred)
    
    def plot_confusion_matrix(self, cm, model_name):
        """Plot confusion matrix."""
        plt.figure(figsize=(8, 6))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                   xticklabels=['Negative', 'Positive'],
                   yticklabels=['Negative', 'Positive'])
        plt.title(f'Confusion Matrix - {model_name}', fontsize=14, fontweight='bold')
        plt.xlabel('Predicted')
        plt.ylabel('Actual')
        plt.show()

print("🔧 Model Evaluator class initialized!")

In [ ]:
# Logistic Regression Pipeline
lr_pipeline = Pipeline([
    ('tfidf', tfidf_vectorizer),
    ('classifier', LogisticRegression(
        random_state=42,
        max_iter=1000,
        C=1.0,
        class_weight='balanced'
    ))
])

# Train and evaluate
lr_evaluator = ModelEvaluator("Logistic Regression")
y_pred_lr, y_pred_proba_lr, lr_model = lr_evaluator.train_and_predict(
    lr_pipeline, X_train, X_test, y_train
)

lr_metrics, lr_cm = lr_evaluator.evaluate_model(y_test, y_pred_lr, y_pred_proba_lr)

print(f"🎯 Logistic Regression Results:")
print(f"Training Time: {lr_evaluator.training_time:.2f}s")
print(f"Prediction Time: {lr_evaluator.prediction_time:.3f}s")
for metric, value in lr_metrics.items():
    print(f"{metric.upper()}: {value:.4f}")

lr_evaluator.plot_confusion_matrix(lr_cm, "Logistic Regression")

# Save the model
joblib.dump(lr_model, 'logistic_regression_model.joblib')
print("💾 Logistic Regression model saved!")

## 🏗️ Model 2: Support Vector Machine

In [ ]:
# SVM Pipeline
svm_pipeline = Pipeline([
    ('tfidf', tfidf_vectorizer),
    ('classifier', SVC(
        random_state=42,
        kernel='linear',
        C=1.0,
        probability=True,
        class_weight='balanced'
    ))
])

# Train and evaluate
svm_evaluator = ModelEvaluator("Support Vector Machine")
y_pred_svm, y_pred_proba_svm, svm_model = svm_evaluator.train_and_predict(
    svm_pipeline, X_train, X_test, y_train
)

svm_metrics, svm_cm = svm_evaluator.evaluate_model(y_test, y_pred_svm, y_pred_proba_svm)

print(f"🎯 Support Vector Machine Results:")
print(f"Training Time: {svm_evaluator.training_time:.2f}s")
print(f"Prediction Time: {svm_evaluator.prediction_time:.3f}s")
for metric, value in svm_metrics.items():
    print(f"{metric.upper()}: {value:.4f}")

svm_evaluator.plot_confusion_matrix(svm_cm, "Support Vector Machine")

# Save the model
joblib.dump(svm_model, 'svm_model.joblib')
print("💾 SVM model saved!")

## 🏗️ Model 3: Random Forest

In [ ]:
# Random Forest Pipeline
rf_pipeline = Pipeline([
    ('tfidf', tfidf_vectorizer),
    ('classifier', RandomForestClassifier(
        random_state=42,
        n_estimators=100,
        max_depth=10,
        min_samples_split=5,
        min_samples_leaf=2,
        class_weight='balanced',
        n_jobs=-1
    ))
])

# Train and evaluate
rf_evaluator = ModelEvaluator("Random Forest")
y_pred_rf, y_pred_proba_rf, rf_model = rf_evaluator.train_and_predict(
    rf_pipeline, X_train, X_test, y_train
)

rf_metrics, rf_cm = rf_evaluator.evaluate_model(y_test, y_pred_rf, y_pred_proba_rf)

print(f"🎯 Random Forest Results:")
print(f"Training Time: {rf_evaluator.training_time:.2f}s")
print(f"Prediction Time: {rf_evaluator.prediction_time:.3f}s")
for metric, value in rf_metrics.items():
    print(f"{metric.upper()}: {value:.4f}")

rf_evaluator.plot_confusion_matrix(rf_cm, "Random Forest")

# Save the model
joblib.dump(rf_model, 'random_forest_model.joblib')
print("💾 Random Forest model saved!")

## 🏗️ Model 4: BERT Fine-tuning

In [ ]:
# BERT Dataset Class
class BERTDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=256):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts.iloc[idx])
        label = int(self.labels.iloc[idx])
        
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

# Initialize BERT tokenizer and model
model_name = 'bert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(model_name)

print(f"🤖 BERT Model: {model_name}")
print(f"🔤 Vocabulary Size: {tokenizer.vocab_size:,}")
print(f"📏 Max Sequence Length: {tokenizer.model_max_length}")

In [ ]:
# Prepare BERT datasets (using smaller subset for faster training)
sample_size = 2000  # Reduce for faster training
X_train_bert = X_train.sample(sample_size, random_state=42)
y_train_bert = y_train.loc[X_train_bert.index]
X_test_bert = X_test.sample(500, random_state=42)
y_test_bert = y_test.loc[X_test_bert.index]

print(f"📊 BERT Training samples: {len(X_train_bert):,}")
print(f"📊 BERT Test samples: {len(X_test_bert):,}")

# Create datasets
train_dataset = BERTDataset(X_train_bert, y_train_bert, tokenizer)
test_dataset = BERTDataset(X_test_bert, y_test_bert, tokenizer)

# Initialize BERT model
bert_model = AutoModelForSequenceClassification.from_pretrained(
    model_name, 
    num_labels=2
)
bert_model.to(device)

print("✅ BERT datasets and model prepared!")

In [ ]:
# Training arguments
training_args = TrainingArguments(
    output_dir='./bert_results',
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    warmup_steps=100,
    weight_decay=0.01,
    logging_dir='./bert_logs',
    logging_steps=50,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='eval_f1',
    greater_is_better=True
)

# Metrics function
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    
    return {
        'accuracy': accuracy_score(labels, predictions),
        'precision': precision_score(labels, predictions),
        'recall': recall_score(labels, predictions),
        'f1': f1_score(labels, predictions)
    }

# Initialize trainer
trainer = Trainer(
    model=bert_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

print("🚀 Starting BERT training...")
start_time = time.time()
trainer.train()
bert_training_time = time.time() - start_time

print(f"✅ BERT training completed in {bert_training_time:.2f}s")

In [ ]:
# Evaluate BERT
bert_eval_results = trainer.evaluate()

# Make predictions
predictions = trainer.predict(test_dataset)
y_pred_bert = np.argmax(predictions.predictions, axis=1)
y_pred_proba_bert = torch.softmax(torch.tensor(predictions.predictions), dim=1)[:, 1].numpy()

# Calculate metrics
bert_metrics = {
    'accuracy': bert_eval_results['eval_accuracy'],
    'precision': bert_eval_results['eval_precision'],
    'recall': bert_eval_results['eval_recall'],
    'f1_score': bert_eval_results['eval_f1']
}

# ROC AUC
fpr, tpr, _ = roc_curve(y_test_bert, y_pred_proba_bert)
bert_metrics['roc_auc'] = auc(fpr, tpr)

# PR AUC
precision, recall, _ = precision_recall_curve(y_test_bert, y_pred_proba_bert)
bert_metrics['pr_auc'] = auc(recall, precision)

bert_cm = confusion_matrix(y_test_bert, y_pred_bert)

print(f"🎯 BERT Results:")
print(f"Training Time: {bert_training_time:.2f}s")
for metric, value in bert_metrics.items():
    print(f"{metric.upper()}: {value:.4f}")

# Plot confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(bert_cm, annot=True, fmt='d', cmap='Blues',
           xticklabels=['Negative', 'Positive'],
           yticklabels=['Negative', 'Positive'])
plt.title('Confusion Matrix - BERT', fontsize=14, fontweight='bold')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

# Save BERT model
bert_model.save_pretrained('bert_sentiment_model')
tokenizer.save_pretrained('bert_sentiment_model')
print("💾 BERT model saved!")

## 📊 Model Comparison & Analysis

In [ ]:
# Create comparison dataframe
models_comparison = pd.DataFrame({
    'Logistic Regression': lr_metrics,
    'SVM': svm_metrics,
    'Random Forest': rf_metrics,
    'BERT': bert_metrics
}).T

# Add training times
models_comparison['training_time'] = [
    lr_evaluator.training_time,
    svm_evaluator.training_time,
    rf_evaluator.training_time,
    bert_training_time
]

print("🏆 MODEL COMPARISON TABLE")
print("=" * 80)
print(models_comparison.round(4))

# Find best model for each metric
print("\n🥇 BEST MODELS BY METRIC:")
for metric in models_comparison.columns[:-1]:  # Exclude training_time
    best_model = models_comparison[metric].idxmax()
    best_score = models_comparison[metric].max()
    print(f"{metric.upper()}: {best_model} ({best_score:.4f})")

In [ ]:
# Visual comparison
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('Model Performance Comparison', fontsize=16, fontweight='bold')

metrics_to_plot = ['accuracy', 'precision', 'recall', 'f1_score', 'roc_auc', 'training_time']
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4']

for idx, metric in enumerate(metrics_to_plot):
    row, col = idx // 3, idx % 3
    ax = axes[row, col]
    
    values = models_comparison[metric].values
    models = models_comparison.index
    
    bars = ax.bar(models, values, color=colors)
    ax.set_title(metric.replace('_', ' ').title(), fontsize=12, fontweight='bold')
    ax.set_ylabel('Score' if metric != 'training_time' else 'Time (seconds)')
    
    # Add value labels on bars
    for bar, value in zip(bars, values):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
               f'{value:.3f}' if metric != 'training_time' else f'{value:.1f}s',
               ha='center', va='bottom', fontsize=10)
    
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 🎯 ROC Curves Comparison

In [ ]:
# Plot ROC curves for all models
plt.figure(figsize=(12, 8))

# Calculate ROC curves for each model
models_data = [
    ('Logistic Regression', y_test, y_pred_proba_lr),
    ('SVM', y_test, y_pred_proba_svm),
    ('Random Forest', y_test, y_pred_proba_rf),
    ('BERT', y_test_bert, y_pred_proba_bert)
]

colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4']

for (model_name, y_true, y_pred_proba), color in zip(models_data, colors):
    fpr, tpr, _ = roc_curve(y_true, y_pred_proba)
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, color=color, linewidth=2,
            label=f'{model_name} (AUC = {roc_auc:.3f})')

plt.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random Classifier')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curves Comparison', fontsize=14, fontweight='bold')
plt.legend(loc="lower right", fontsize=11)
plt.grid(True, alpha=0.3)
plt.show()

## 💾 Save Best Model & Results

In [ ]:
# Determine best model based on F1-score
best_model_name = models_comparison['f1_score'].idxmax()
best_f1_score = models_comparison['f1_score'].max()

print(f"🏆 Best Model: {best_model_name}")
print(f"📊 Best F1-Score: {best_f1_score:.4f}")

# Save comparison results
models_comparison.to_csv('model_comparison_results.csv')
print("\n💾 Model comparison results saved to 'model_comparison_results.csv'")

# Save best model info
best_model_info = {
    'model_name': best_model_name,
    'f1_score': best_f1_score,
    'all_metrics': models_comparison.loc[best_model_name].to_dict()
}

with open('best_model_info.pkl', 'wb') as f:
    pickle.dump(best_model_info, f)

print("💾 Best model information saved!")

print("\n✅ Model development phase completed!")
print("🚀 Ready for API deployment phase!")